# VIX Distribution Properties

This notebook studies the descriptive distribution of daily VIX Close and monthly VIX High. It compares raw and log-transformed values, checks Q-Q behavior, and examines stability across market regimes.

Strategy-specific thresholds and wave rules are documented in the [VIX event strategy](../04_vix_event_strategy/README.md).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import kurtosis, shapiro, skew


def distribution_stats(series, name):
    clean = series.dropna()
    sample = clean.sample(5000, random_state=42) if len(clean) > 5000 else clean
    shapiro_stat, shapiro_p = shapiro(sample)
    return pd.Series({
        "Name": name,
        "Count": len(clean),
        "Mean": clean.mean(),
        "Std": clean.std(),
        "Median": clean.median(),
        "Min": clean.min(),
        "Max": clean.max(),
        "Skewness": skew(clean),
        "Kurtosis_Excess": kurtosis(clean),
        "Shapiro_Stat": shapiro_stat,
        "Shapiro_p": shapiro_p,
    })


## Data preparation

Daily Close represents the commonly observed VIX level. Monthly High captures the upper stress level reached within each month; here it is a distribution variable, not an event signal.


In [ ]:
vix = yf.download(
    "^VIX",
    start="1990-01-01",
    auto_adjust=False,
    progress=False,
)

daily_close = vix["Close"].squeeze().rename("VIX_Close").dropna()
monthly_high = vix["High"].squeeze().resample("ME").max().rename("VIX_High").dropna()

distribution_data = pd.DataFrame({
    "Daily_Close": daily_close,
    "Log_Daily_Close": np.log(daily_close),
})

monthly_data = pd.DataFrame({
    "Monthly_High": monthly_high,
    "Log_Monthly_High": np.log(monthly_high),
})

print(f"Daily sample: {daily_close.index.min().date()} to {daily_close.index.max().date()}")
print(f"Monthly sample: {monthly_high.index.min().date()} to {monthly_high.index.max().date()}")


## Raw and log distribution comparison


In [ ]:
summary = pd.DataFrame([
    distribution_stats(distribution_data["Daily_Close"], "Raw daily VIX Close"),
    distribution_stats(distribution_data["Log_Daily_Close"], "Log daily VIX Close"),
    distribution_stats(monthly_data["Monthly_High"], "Raw monthly VIX High"),
    distribution_stats(monthly_data["Log_Monthly_High"], "Log monthly VIX High"),
])

display(summary)


In [ ]:
series_to_plot = [
    (distribution_data["Daily_Close"], "Raw daily VIX Close"),
    (distribution_data["Log_Daily_Close"], "Log daily VIX Close"),
    (monthly_data["Monthly_High"], "Raw monthly VIX High"),
    (monthly_data["Log_Monthly_High"], "Log monthly VIX High"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (series, title) in zip(axes.flat, series_to_plot):
    clean = series.dropna()
    axis.hist(clean, bins=50, density=True, alpha=0.65)
    x = np.linspace(clean.min(), clean.max(), 300)
    axis.plot(x, stats.norm.pdf(x, clean.mean(), clean.std()), linewidth=2)
    axis.set_title(title)
    axis.set_ylabel("Density")

plt.tight_layout()
plt.show()


## Q-Q plots

The plots test the shape of the center and tails rather than assuming a perfect normal or lognormal model.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (series, title) in zip(axes.flat, series_to_plot):
    stats.probplot(series.dropna(), dist="norm", plot=axis)
    axis.set_title(f"Q-Q plot: {title}")

plt.tight_layout()
plt.show()


## Regime comparison of monthly VIX High


In [ ]:
periods = {
    "1995-2007": ("1995-01-01", "2007-12-31"),
    "2008-2019": ("2008-01-01", "2019-12-31"),
    "2020-present": ("2020-01-01", None),
}

regime_rows = []
for name, (start, end) in periods.items():
    subset = monthly_data.loc[start:end, "Log_Monthly_High"]
    if len(subset) >= 10:
        regime_rows.append(distribution_stats(subset, name))

regime_summary = pd.DataFrame(regime_rows)
display(regime_summary)

fig, axes = plt.subplots(1, len(regime_rows), figsize=(16, 5))
for axis, row in zip(np.atleast_1d(axes), regime_rows):
    name = row["Name"]
    start, end = periods[name]
    subset = monthly_data.loc[start:end, "Log_Monthly_High"].dropna()
    stats.probplot(subset, dist="norm", plot=axis)
    axis.set_title(f"{name}: log monthly High")

plt.tight_layout()
plt.show()


## Interpretation boundary

This notebook describes the empirical distribution. It does not define stress-event dates or trading entries. Those depend on the separate event-strategy methodology.
